**Note added on 2026-11-22.** The gpt-audio request in this study placed the audio part before the text part. A control on 2026-11-22 showed that gpt-audio ignores the audio in that order. The gpt-audio results from this notebook record that failure. Study 1A was rerun for gpt-audio with the text part first. The control and the rerun are in `../gpt-audio control and rerun (2026-11-22)`. The code in this notebook is unchanged.

# Study 1: Age Estimation from Children's Speech (Audio)

28 children, ages 5 to 10, three scripted utterances each, about 6s of audio per child.

**Conditions (4):**
- Gemini 2.5 Flash run under `google-genai` 0.4.0, the version that corrupted inline audio
- Gemini 2.5 Flash run under `google-genai` 2.0.0, the version that fixed it
- gpt-audio (OpenAI), successor to the retired gpt-4o-audio-preview
- Nemotron 3 Nano Omni (NVIDIA) via fal.ai

**Questions:** Q1 over 13, Q2 over 18, Q3 estimate age with reasoning.

**Repeats:** 3

The two Gemini conditions run the identical call code. Only the installed SDK version differs between them. Section 8 shows why one is accurate and the other is not.

## 1. Setup

In [ ]:
# This notebook expects the 'study1-pinned' kernel (Python 3.13, pinned packages).
# The version check below confirms the core pins before anything runs.
from importlib.metadata import version, PackageNotFoundError

REQUIRED = {
    "google-genai": "2.0.0", "openai": "2.36.0", "fal_client": "1.0.0",
    "numpy": "2.4.4", "pandas": "3.0.2", "scipy": "1.17.1", "soundfile": "0.13.1",
}

def _check():
    bad = 0
    for name, want in REQUIRED.items():
        try:
            if version(name) != want:
                bad += 1
        except PackageNotFoundError:
            bad += 1
    return bad

_bad = _check()
print("Versions match" if _bad == 0 else f"Versions don't match ({_bad} off)")

In [ ]:
import os
import pandas as pd
import numpy as np
import base64
import time
import re
import json
import io
import warnings
import subprocess
import venv
import soundfile as sf
from pathlib import Path
from getpass import getpass
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

STIMULI_DIR = Path('study1_stimuli')
N_REPEATS = 3
RATE_LIMIT_DELAY = 1.0

# Set to an integer to run a quick subset, or None for the full 28 children.
MAX_STIMULI = None

## 2. API Keys

In [ ]:
GEMINI_API_KEY = getpass("Google Gemini API key: ")
OPENAI_API_KEY = getpass("OpenAI API key: ")
FAL_API_KEY = getpass("fal.ai API key: ")

# fal uses an environment variable
os.environ['FAL_KEY'] = FAL_API_KEY

print("Gemini:", "set" if GEMINI_API_KEY else "MISSING")
print("OpenAI:", "set" if OPENAI_API_KEY else "MISSING")
print("fal.ai:", "set" if FAL_API_KEY else "MISSING")

## 3. SDK environments for the Gemini version comparison

A single Python process cannot hold two versions of `google-genai`. The kernel stays on 2.0.0.
The 0.4.0 calls run inside a separate virtual environment, reached through a subprocess.
These environments are built once and reused.

In [ ]:
ENVS = Path("demo_envs")
SPECS = {"v0_4_0": "google-genai==0.4.0", "v2_0_0": "google-genai==2.0.0"}

def ensure_env(name, spec):
    py = ENVS / name / "bin" / "python"
    if py.exists():
        return py
    venv.create(ENVS / name, with_pip=True)
    subprocess.run([str(py), "-m", "pip", "install", "-q", spec, "httpx"], check=True)
    return py

PYTHONS = {name: ensure_env(name, spec) for name, spec in SPECS.items()}
for name, py in PYTHONS.items():
    print(name, "->", py)

In [ ]:
worker_src = r'''
import sys, os, json, base64, gzip, zlib, time
import re as _re

JOBS = sys.argv[1]
OUT  = sys.argv[2]
API_KEY = os.environ["GEMINI_API_KEY"]
MODEL   = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")

# Record every outgoing request body, across both HTTP libraries the SDK might use.
CAPTURED = []

try:
    import httpx
    _h_send = httpx.Client.send
    def _h_spy(self, request, *a, **k):
        try: CAPTURED.append(request.content)
        except Exception: pass
        return _h_send(self, request, *a, **k)
    httpx.Client.send = _h_spy
except Exception:
    pass

try:
    import requests
    _r_send = requests.sessions.Session.send
    def _r_spy(self, request, *a, **k):
        try: CAPTURED.append(request.body)
        except Exception: pass
        return _r_send(self, request, *a, **k)
    requests.sessions.Session.send = _r_spy
except Exception:
    pass

import google.genai as genai
from importlib.metadata import version as _v
sdk_version = _v("google-genai")

client = genai.Client(api_key=API_KEY)
with open(JOBS) as jf:
    jobs = json.loads(jf.read())


def run_one(audio_path, question):
    with open(audio_path, "rb") as f:
        audio_bytes = f.read()
    flac_b64 = base64.b64encode(audio_bytes).decode("utf-8")
    # ===== verbatim original call_gemini path =====
    resp = client.models.generate_content(
        model=MODEL,
        contents=[
            {"inline_data": {"mime_type": "audio/flac", "data": flac_b64}},
            question,
        ],
    )
    return resp.text.strip()
    # ===== end call block =====


def decompress(raw):
    if isinstance(raw, str):
        raw = raw.encode("utf-8", "ignore")
    if not isinstance(raw, (bytes, bytearray)):
        return b""
    raw = bytes(raw)
    for attempt in (lambda b: gzip.decompress(b),
                    lambda b: zlib.decompress(b),
                    lambda b: zlib.decompress(b, -zlib.MAX_WBITS),
                    lambda b: b):
        try:
            return attempt(raw)
        except Exception:
            continue
    return raw


# Longest base64-ish run in the raw body, standard or url-safe, padded or not.
_b64_run = _re.compile(rb"[A-Za-z0-9+/_-]{200,}={0,2}")


def best_audio_field(bodies):
    best = b""
    for body in bodies:
        if body is None:
            continue
        text = decompress(body)
        for m in _b64_run.findall(text):
            if len(m) > len(best):
                best = m
    return best


def try_decode(b):
    if isinstance(b, (bytes, bytearray)):
        s = bytes(b).decode("ascii", "ignore")
    else:
        s = str(b)
    s = s.replace("-", "+").replace("_", "/")
    s += "=" * (-len(s) % 4)
    return base64.b64decode(s)


with open(OUT, "w") as out:
    for i, job in enumerate(jobs):
        try:
            ans = run_one(job["audio_path"], job["question"])
        except Exception as e:
            ans = f"ERROR: {e}"
        out.write(json.dumps({"tag": job["tag"], "response": ans}) + "\n")
        out.flush()
        print(f"PROGRESS {i+1}/{len(jobs)}", file=sys.stderr, flush=True)
        time.sleep(0.5)

    forensic = {"captured": False, "bodies_seen": len(CAPTURED)}
    field = best_audio_field(CAPTURED)
    if field:
        try:
            dec = try_decode(field)
            forensic = {
                "captured": True,
                "bodies_seen": len(CAPTURED),
                "wire_b64_len": len(field),
                "decoded_len": len(dec),
                "decoded_head_ascii": dec[:8].decode("latin1"),
                "decoded_is_flac": dec[:4] == b"fLaC",
            }
            if dec[:4] != b"fLaC":
                try:
                    d2 = try_decode(dec)
                    forensic["double_decoded_is_flac"] = d2[:4] == b"fLaC"
                except Exception:
                    forensic["double_decoded_is_flac"] = False
        except Exception as e:
            forensic = {"captured": False, "bodies_seen": len(CAPTURED), "decode_error": repr(e)}

    out.write(json.dumps({"type": "meta", "sdk_version": sdk_version, "forensic": forensic}) + "\n")
'''
Path("gemini_batch_worker.py").write_text(worker_src)
print("wrote gemini_batch_worker.py")

## 4. Load Stimuli

In [ ]:
with open(STIMULI_DIR / 'manifest.json') as f:
    manifest = json.load(f)

if MAX_STIMULI:
    manifest = manifest[:MAX_STIMULI]

for entry in manifest:
    audio_path = STIMULI_DIR / entry['audio_file']

    # FLAC bytes + base64 (Gemini)
    with open(audio_path, 'rb') as f:
        entry['flac_bytes'] = f.read()
    entry['flac_b64'] = base64.b64encode(entry['flac_bytes']).decode('utf-8')

    # WAV bytes + base64 (gpt-audio)
    data, sr = sf.read(audio_path)
    wav_buf = io.BytesIO()
    sf.write(wav_buf, data, sr, format='WAV')
    wav_buf.seek(0)
    entry['wav_bytes'] = wav_buf.read()
    entry['wav_b64'] = base64.b64encode(entry['wav_bytes']).decode('utf-8')

    print(f'  {entry["child"]:>6s}: age {entry["age"]:>2d}, {entry["gender"]}, {entry["duration"]:.1f}s')

print(f'\nLoaded {len(manifest)} stimuli.')
from collections import Counter
for age, n in sorted(Counter(e['age'] for e in manifest).items()):
    print(f'  Age {age}: {n}')

## 5. Define Questions

In [ ]:
QUESTIONS = {
    'q1_over_13': {
        'text': (
            'Listen to this audio clip. Based on the acoustic and linguistic '
            'features of the speech, is the speaker over 13 years old?\n\n'
            'Respond with YES or NO, followed by a brief one-sentence reasoning.'
        ),
        'label': 'Over 13?',
    },
    'q2_over_18': {
        'text': (
            'Listen to this audio clip. Based on the acoustic and linguistic '
            'features of the speech, is the speaker over 18 years old?\n\n'
            'Respond with YES or NO, followed by a brief one-sentence reasoning.'
        ),
        'label': 'Over 18?',
    },
    'q3_age': {
        'text': (
            'Listen to this audio clip. Based on the acoustic and linguistic '
            'features of the speech, estimate the speaker\'s age in years.\n\n'
            'Respond in this format:\n'
            'Estimated age: [number]\n'
            'Confidence: [low/medium/high]\n'
            'Brief reasoning: [one sentence about aspects of the pitch, speech rate, '
            'articulation, vocabulary, sentence complexity, and any other relevant '
            'speech patterns that led to your estimation]'
        ),
        'label': 'Age estimate',
    },
}

print('Questions defined:', list(QUESTIONS.keys()))

## 6. Define Model Conditions

In [ ]:
def run_gemini_batch(entries, questions, n_repeats, env_name):
    """Run all Gemini jobs under one SDK version in a single subprocess launch, streaming progress."""
    py = PYTHONS[env_name]
    jobs = []
    for e in entries:
        for qk, qi in questions.items():
            for rep in range(n_repeats):
                jobs.append({
                    'tag': f"{e['child']}|{qk}|{rep+1}",
                    'audio_path': str(STIMULI_DIR / e['audio_file']),
                    'question': qi['text'],
                })
    job_path = f'gemini_jobs_{env_name}.json'
    out_path = f'gemini_out_{env_name}.jsonl'
    Path(job_path).write_text(json.dumps(jobs))
    env = dict(os.environ, GEMINI_API_KEY=GEMINI_API_KEY, GEMINI_MODEL='gemini-2.5-flash')

    t_start = time.time()
    proc = subprocess.Popen(
        [str(py), 'gemini_batch_worker.py', job_path, out_path],
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True, env=env)

    err_lines = []
    for line in proc.stderr:
        line = line.rstrip('\n')
        if line.startswith('PROGRESS '):
            i, n = line[len('PROGRESS '):].split('/')
            i, n = int(i), int(n)
            elapsed = time.time() - t_start
            rate = i / elapsed if elapsed > 0 else 0
            left = (n - i) / rate / 60 if rate > 0 else float('nan')
            print(f'\r   gemini {i}/{n} | {rate:.1f} calls/s | ~{left:.0f} min left   ',
                  end='', flush=True)
        else:
            err_lines.append(line)
    proc.wait()
    print()

    if proc.returncode != 0:
        raise RuntimeError('\n'.join(err_lines) or f'worker exited {proc.returncode}')

    rows, forensic, sdk_version = [], None, None
    for line in Path(out_path).read_text().splitlines():
        rec = json.loads(line)
        if rec.get('type') == 'meta':
            forensic = rec.get('forensic'); sdk_version = rec.get('sdk_version'); continue
        child, qk, rep = rec['tag'].split('|')
        rows.append({'child': child, 'question': qk, 'repeat': int(rep), 'response': rec['response']})
    return rows, forensic, sdk_version

## 7. The serialization bug, side by side

Before audio can travel to Google's servers it has to be packed into plain text. We can think of that packing as sealing the audio inside an envelope. The correct version of the software sealed it once. The broken version sealed it twice. It put one envelope inside another.
The server only ever opens the outer envelope. With the fixed version, it opened that envelope and found the audio, so Gemini could listen and estimate an age. With the broken version it opened the envelope and found a second sealed envelope where the audio should have been. Gemini had nothing to listen to, so it guessed that every child was an adult.

The cell below sends the exact same request twice. It sends it once under the broken version and once under the fixed version. It then reports what each one actually delivered. The column for whether the audio was readable right away says yes for the fixed version and no for the broken one. The size column shows the broken version is about a third larger. That extra bulk is what you get whenever the same thing is packed twice.

You do not need to read any of the other numbers. The two rows should look plainly different. That difference is the bug.

In [ ]:
demo_entry = [manifest[0]]
demo_q = {'q3_age': QUESTIONS['q3_age']}
rows040, forensic040, ver040 = run_gemini_batch(demo_entry, demo_q, 1, 'v0_4_0')
rows200, forensic200, ver200 = run_gemini_batch(demo_entry, demo_q, 1, 'v2_0_0')

def _yn(v):
    return {True: 'yes', False: 'no'}.get(v, 'not captured')

def _first_line(rows):
    if not rows:
        return None
    return rows[0]['response'].strip().split('\n')[0].strip()[:60]

def _row(ver, forensic, rows):
    f = forensic or {}
    if f.get('captured'):
        size = f"{f.get('wire_b64_len'):,}"
        ready = _yn(f.get('decoded_is_flac'))
        second = _yn(f.get('double_decoded_is_flac', False))
    else:
        size = ready = second = 'not captured'
    return {
        'google-genai version': ver,
        'packed size in characters': size,
        'audio readable right away': ready,
        'readable only after a second unpacking': second,
        "Gemini's answer": _first_line(rows),
    }

demo_tbl = pd.DataFrame([_row(ver040, forensic040, rows040),
                         _row(ver200, forensic200, rows200)]).set_index('google-genai version')

if forensic040.get('wire_b64_len') and forensic200.get('wire_b64_len'):
    ratio = forensic040['wire_b64_len'] / forensic200['wire_b64_len']
    print(f"The broken version sent {ratio:.2f} times more data than the fixed version.")
    print("A value near 1.33 is the sign of packing the same audio twice.\n")

demo_tbl

In [ ]:
REG = Path('google_genai regression')

def forensic(fname):
    for line in Path(REG / fname).read_text().splitlines():
        rec = json.loads(line)
        if rec.get('type') == 'meta':
            return rec
    return None

def first_age(fname):
    for line in Path(REG / fname).read_text().splitlines():
        rec = json.loads(line)
        if rec.get('tag', '').endswith('q3_age|1'):
            return rec['response'].splitlines()[0].strip()
    return None

rows = []
for fname, ver in [('gemini_out_v0_4_0.jsonl', '0.4.0'),
                   ('gemini_out_v2_0_0.jsonl', '2.0.0')]:
    f = forensic(fname)['forensic']
    rows.append({
        'google-genai version': ver,
        'Packed size in characters': f'{f["wire_b64_len"]:,}',
        'Audio readable on first decode': 'yes' if f['decoded_is_flac'] else 'no',
        'Readable only after a second decode': 'yes' if f.get('double_decoded_is_flac') else 'no',
        "Gemini's estimate": first_age(fname),
    })

demo_tbl = pd.DataFrame(rows).set_index('google-genai version')
demo_tbl

## 8. Quick Test (1 child, all conditions)

In [ ]:
test_child = manifest[0]
test_q = QUESTIONS['q3_age']['text']
print(f'Testing: {test_child["child"]} (age {test_child["age"]})\n')

# Gemini both versions, one call each
for mk in ['gemini_040', 'gemini_200']:
    t0 = time.time()
    rows, _, _ = run_gemini_batch([test_child], {'q3_age': QUESTIONS['q3_age']}, 1, MODELS[mk]['env'])
    dt = time.time() - t0
    resp = rows[0]['response'] if rows else 'ERROR: no response'
    status = 'ERROR' if resp.startswith('ERROR') else 'OK'
    print(f'{MODELS[mk]["label"]:.<34s} {status} ({dt:.1f}s)')
    print(f'  {resp[:250]}\n')

# in-kernel conditions
for mk in ['gpt_audio', 'nemotron']:
    t0 = time.time()
    resp = MODELS[mk]['caller'](test_child, test_q)
    dt = time.time() - t0
    status = 'ERROR' if resp.startswith('ERROR') else 'OK'
    print(f'{MODELS[mk]["label"]:.<34s} {status} ({dt:.1f}s)')
    print(f'  {resp[:250]}\n')

## 9. Run Full Experiment (3 conditions)

In [ ]:
import sys

results = []
t0 = time.time()
meta = {e['child']: e for e in manifest}

n_stim = len(manifest)
gemini_conditions = ['gemini_200']
inkernel = ['gpt_audio', 'nemotron']

gemini_calls = len(gemini_conditions) * n_stim * len(QUESTIONS) * N_REPEATS
inkernel_calls = len(inkernel) * n_stim * len(QUESTIONS) * N_REPEATS
total_calls = gemini_calls + inkernel_calls

print(f"Total API calls planned: {total_calls}")
print(f"  Gemini 2.0.0 (batched) : {gemini_calls}")
print(f"  In-kernel (gpt+nemo)   : {inkernel_calls}")
print("-" * 50)

# --- Gemini condition via batched worker ---
gemini_forensics = {}
done = 0
for mk in gemini_conditions:
    print(f'Running {MODELS[mk]["label"]} ...')
    rows, forensic, sdk_version = run_gemini_batch(manifest, QUESTIONS, N_REPEATS, MODELS[mk]['env'])
    gemini_forensics[mk] = {'forensic': forensic, 'sdk_version': sdk_version}
    for r in rows:
        e = meta[r['child']]
        results.append({
            'child': e['child'], 'actual_age': e['age'], 'gender': e['gender'],
            'duration': e['duration'], 'model': mk, 'model_label': MODELS[mk]['label'],
            'question': r['question'], 'question_label': QUESTIONS[r['question']]['label'],
            'repeat': r['repeat'], 'response': r['response'],
        })
    done += len(rows)
    elapsed = time.time() - t0
    rate = done / elapsed if elapsed > 0 else 0
    remaining = (total_calls - done) / rate / 60 if rate > 0 else float('nan')
    print(f'  {len(rows)} responses (genai {sdk_version}) | '
          f'{done}/{total_calls} done | ~{remaining:.0f} min left')

# --- in-kernel conditions ---
ik_done = 0
last_print = 0.0
for entry in manifest:
    for mk in inkernel:
        for qk, qi in QUESTIONS.items():
            for rep in range(N_REPEATS):
                resp = MODELS[mk]['caller'](entry, qi['text'])
                results.append({
                    'child': entry['child'], 'actual_age': entry['age'],
                    'gender': entry['gender'], 'duration': entry['duration'],
                    'model': mk, 'model_label': MODELS[mk]['label'],
                    'question': qk, 'question_label': qi['label'],
                    'repeat': rep + 1, 'response': resp,
                })
                done += 1
                ik_done += 1
                time.sleep(RATE_LIMIT_DELAY)

                now = time.time()
                if now - last_print > 2 or ik_done == inkernel_calls:
                    last_print = now
                    elapsed = now - t0
                    rate = done / elapsed if elapsed > 0 else 0
                    remaining = (total_calls - done) / rate / 60 if rate > 0 else float('nan')
                    print(f'\r  in-kernel {ik_done}/{inkernel_calls} | '
                          f'total {done}/{total_calls} | '
                          f'{rate:.1f} calls/s | ~{remaining:.0f} min left   ',
                          end='', flush=True)
print()

results_df = pd.DataFrame(results)
print(f'\nDone. {len(results_df)} total results in {(time.time()-t0)/60:.1f} min.')

## 10. Diagnostics

In [ ]:
for mk in MODELS:
    sub = results_df[results_df['model'] == mk]
    errs = sub[sub['response'].str.startswith('ERROR')]
    refs = sub[sub['response'].str.contains("sorry|can't estimate|can't determine|can't identify|can't analyze", case=False, na=False)]
    ok = sub[~sub.index.isin(errs.index) & ~sub.index.isin(refs.index)]
    print(f'{MODELS[mk]["label"]:.<34s} Errors:{len(errs):>4d}  Refusals:{len(refs):>4d}  Answers:{len(ok):>4d}')
    if len(errs) > 0:
        print(f'  Error: {errs.iloc[0]["response"][:200]}')
    if len(refs) > 0:
        print(f'  Refusal: {refs.iloc[0]["response"][:200]}')

## 11. gpt-audio Refusal Analysis

In [ ]:
gpt_q3 = results_df[(results_df['model'] == 'gpt_audio') & (results_df['question'] == 'q3_age')].copy()
gpt_q3['refused'] = (
    gpt_q3['response'].str.contains("sorry|can't estimate|can't determine|can't identify|can't analyze", case=False, na=False) |
    gpt_q3['response'].str.startswith('ERROR'))

print('gpt-audio refusal rate by age:')
for age in sorted(gpt_q3['actual_age'].unique()):
    s = gpt_q3[gpt_q3['actual_age'] == age]
    print(f'  Age {age}: {100*s["refused"].mean():.0f}% refused (n_kids={s["child"].nunique()})')

cr = gpt_q3.groupby(['child','actual_age'])['refused'].mean().reset_index()
if cr['refused'].nunique() > 1 and len(cr) > 2:
    r, p = stats.pearsonr(cr['actual_age'], cr['refused'])
    print(f'\nAge-refusal correlation: r={r:.3f}, p={p:.4f}')
else:
    print('\nRefusal is uniform across ages, so no age correlation is computed.')

## 12. Parse Responses

In [ ]:
def parse_yn(r):
    if r.startswith('ERROR'): return 'ERROR'
    if re.search(r"sorry|can't", r, re.I): return 'REFUSED'
    u = r.strip().upper()
    if u.startswith('YES'): return 'YES'
    if u.startswith('NO'): return 'NO'
    fl = u.split('\n')[0]
    if 'YES' in fl and 'NO' not in fl: return 'YES'
    if 'NO' in fl and 'YES' not in fl: return 'NO'
    return 'UNCLEAR'

def parse_age(r):
    if r.startswith('ERROR'): return None
    if re.search(r"sorry|can't estimate|can't determine|can't identify|can't analyze", r, re.I): return None
    m = re.search(r'[Ee]stimated\s+[Aa]ge[:\s]+(\d+)', r)
    if m: return int(m.group(1))
    m = re.search(r'(\d+)\s*-\s*(\d+)\s*years', r)
    if m: return (int(m.group(1)) + int(m.group(2))) / 2
    nums = re.findall(r'\d+', r.split('\n')[0])
    return int(nums[0]) if nums else None

q1 = results_df[results_df['question'] == 'q1_over_13'].copy()
q1['pred'] = q1['response'].apply(parse_yn)
q2 = results_df[results_df['question'] == 'q2_over_18'].copy()
q2['pred'] = q2['response'].apply(parse_yn)
q3 = results_df[results_df['question'] == 'q3_age'].copy()
q3['pred_age'] = q3['response'].apply(parse_age)
q3['signed_error'] = q3['pred_age'] - q3['actual_age']
valid_q3 = q3.dropna(subset=['pred_age'])

print('Valid Q3 predictions per condition:')
for mk in MODELS:
    s = valid_q3[valid_q3['model'] == mk]
    print(f'  {MODELS[mk]["label"]}: {len(s)} from {s["child"].nunique()} children')

## 13. Binary Classification (Q1, Q2)

In [ ]:
for label, qdf in [("Q1: Over 13? (correct: NO)", q1), ("Q2: Over 18? (correct: NO)", q2)]:
    print(label)
    print('=' * 60)
    for mk in MODELS:
        s = qdf[(qdf['model'] == mk) & qdf['pred'].isin(['YES','NO'])]
        if len(s) == 0:
            print(f'  {MODELS[mk]["label"]}: no valid responses'); continue
        print(f'  {MODELS[mk]["label"]}:')
        for age in sorted(s['actual_age'].unique()):
            a = s[s['actual_age'] == age]
            print(f'    Age {age} (n={a["child"].nunique()}): {100*(a["pred"]=="YES").mean():.0f}% YES')
    print()

## 14. Age Estimation (Q3)

### 14a. Overall bias

In [ ]:
for mk in MODELS:
    s = valid_q3[valid_q3['model'] == mk]
    if len(s) == 0:
        print(f'{MODELS[mk]["label"]}: no valid predictions'); continue
    e = s['signed_error']
    cm = s.groupby('child')['signed_error'].mean()
    print(f'{MODELS[mk]["label"]} ({s["child"].nunique()} children):')
    print(f'  Mean error: {e.mean():+.2f} | Median: {e.median():+.1f} | MAE: {e.abs().mean():.2f}')
    if len(cm) >= 3:
        t, p = stats.ttest_1samp(cm, 0)
        sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'n.s.'
        print(f'  t={t:.3f}, p={p:.4f} {sig}')
    print()

### 14b. By age

In [ ]:
for mk in MODELS:
    s = valid_q3[valid_q3['model'] == mk]
    if len(s) == 0: continue
    print(f'{MODELS[mk]["label"]}:')
    print(f'  {"Age":>4s} {"n":>3s} {"Est":>6s} {"Err":>6s}')
    for age in sorted(s['actual_age'].unique()):
        a = s[s['actual_age'] == age]
        print(f'  {age:>4d} {a["child"].nunique():>3d} {a["pred_age"].mean():>6.1f} {a["signed_error"].mean():>+6.1f}')
    print()

### 14c. Compression test

In [ ]:
for mk in MODELS:
    s = valid_q3[valid_q3['model'] == mk]
    cm = s.groupby('child').agg(age=('actual_age','first'), err=('signed_error','mean'))
    if len(cm) < 3:
        print(f'{MODELS[mk]["label"]}: skipping ({len(cm)} children)'); continue
    r, p = stats.pearsonr(cm['age'], cm['err'])
    sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'n.s.'
    print(f'{MODELS[mk]["label"]}: r={r:.3f}, p={p:.4f} {sig}')
    print(f'  {"=> Compression" if r < 0 else "=> No compression"}')

### 14d. Per-child

In [ ]:
active = [mk for mk in MODELS if len(valid_q3[valid_q3['model']==mk]) > 0]
header = f'{"Child":>7s} {"Age":>4s} {"G":>2s}'
for mk in active: header += f' {MODELS[mk]["short"]:>10s}'
print(header)
print('-' * len(header))
for e in sorted(manifest, key=lambda x: (x['age'], x['child'])):
    line = f'{e["child"]:>7s} {e["age"]:>4d} {e["gender"]:>2s}'
    for mk in active:
        s = valid_q3[(valid_q3['model']==mk) & (valid_q3['child']==e['child'])]
        if len(s) > 0:
            line += f' {s["pred_age"].mean():>5.1f}({s["signed_error"].mean():+4.1f})'
        else:
            line += f' {"refused":>10s}'
    print(line)

## 15. Visualization

### 15a. Predicted vs. actual

In [ ]:
active = [mk for mk in MODELS if len(valid_q3[valid_q3['model']==mk]) > 0]
colors = {mk: MODELS[mk]['color'] for mk in MODELS}
fig, axes = plt.subplots(1, len(active), figsize=(6*len(active), 5))
if len(active)==1: axes=[axes]
for i, mk in enumerate(active):
    ax = axes[i]
    s = valid_q3[valid_q3['model']==mk]
    cs = s.groupby('child').agg(age=('actual_age','first'), est=('pred_age','mean')).reset_index()
    j = np.random.default_rng(42).normal(0, 0.1, len(cs))
    ax.scatter(cs['age']+j, cs['est'], c=colors.get(mk,'gray'), s=80, edgecolor='white', alpha=.8)
    ax.plot([3,50],[3,50],'k--',lw=1,alpha=.4)
    ax.set_xlabel('Actual Age'); ax.set_ylabel('Predicted Age')
    ax.set_title(f'{MODELS[mk]["short"]} ({len(cs)} children)')
    ax.set_xlim(3,13)
sns.despine(); plt.tight_layout()
plt.savefig('study1_pred_vs_actual.png', dpi=150, bbox_inches='tight'); plt.show()

### 15b. Error by age

In [ ]:
fig, axes = plt.subplots(1, len(active), figsize=(6*len(active), 5))
if len(active)==1: axes=[axes]
for i, mk in enumerate(active):
    ax = axes[i]
    s = valid_q3[valid_q3['model']==mk]
    ages = sorted(s['actual_age'].unique())
    bp = ax.boxplot([s[s['actual_age']==a]['signed_error'].values for a in ages],
                     tick_labels=ages, patch_artist=True, medianprops={'color':'black','linewidth':1.5})
    for p in bp['boxes']: p.set_facecolor(colors.get(mk,'gray')); p.set_alpha(.6)
    ax.axhline(y=0,color='black',linestyle='--',lw=1,alpha=.5)
    ax.set_xlabel('Actual Age'); ax.set_ylabel('Signed Error')
    ax.set_title(MODELS[mk]['short'])
sns.despine(); plt.tight_layout()
plt.savefig('study1_error_by_age.png', dpi=150, bbox_inches='tight'); plt.show()

### 15c. Model comparison

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ages = sorted(valid_q3['actual_age'].unique())
x = np.arange(len(ages))
w = 0.8 / max(len(active),1)
for i, mk in enumerate(active):
    s = valid_q3[valid_q3['model']==mk]
    means = [s[s['actual_age']==a]['signed_error'].mean() if len(s[s['actual_age']==a])>0 else 0 for a in ages]
    sems = [s[s['actual_age']==a]['signed_error'].sem() if len(s[s['actual_age']==a])>1 else 0 for a in ages]
    ax.bar(x+(i-len(active)/2+.5)*w, means, w, yerr=sems, label=MODELS[mk]['short'],
           color=colors.get(mk,'gray'), alpha=.7, edgecolor='white', capsize=3)
ax.axhline(y=0,color='black',linestyle='--',lw=1,alpha=.5)
ax.set_xticks(x); ax.set_xticklabels(ages)
ax.set_xlabel('Actual Age'); ax.set_ylabel('Mean Signed Error')
ax.set_title('Condition Comparison'); ax.legend()
sns.despine(); plt.tight_layout()
plt.savefig('study1_model_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

### 15d. gpt-audio refusal rate

In [ ]:
gpt_all = results_df[results_df['model']=='gpt_audio'].copy()
gpt_all['refused'] = (gpt_all['response'].str.contains("sorry|can't estimate|can't determine|can't identify|can't analyze",case=False,na=False) | gpt_all['response'].str.startswith('ERROR'))
rba = gpt_all.groupby('actual_age')['refused'].mean()
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(rba.index, rba.values*100, color='coral', alpha=.7, edgecolor='white')
ax.set_xlabel('Actual Age'); ax.set_ylabel('Refusal Rate (%)')
ax.set_title('gpt-audio Refusal Rate by Age'); ax.set_ylim(0,105)
sns.despine(); plt.tight_layout()
plt.savefig('study1_gpt_audio_refusal.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# GPT refusal right by candidate model: are there other audio-capable OpenAI models that would refuse less than `gpt-audio-2025-08-28`?
# TL;DR: No.

candidates = ["gpt-audio", "gpt-audio-1.5", "gpt-audio-2025-08-28", "gpt-audio-mini-2025-10-06"]

# one child per age, 5 through 10
probe, seen = [], set()
for e in sorted(manifest, key=lambda x: x['age']):
    if e['age'] not in seen:
        probe.append(e); seen.add(e['age'])

q = QUESTIONS['q3_age']['text']

def classify(text):
    if text.startswith('ERROR'): return 'error'
    if re.search(r"provide the audio|no audio|isn't any audio|don't have access", text, re.I):
        return 'no-audio'
    if re.search(r"sorry|can't|cannot|unable|not able|can not", text, re.I):
        return 'refused'
    a = parse_age(text)
    return f'age={a}' if a is not None else 'answered-no-age'

for model in candidates:
    print(f"\n=== {model} ===")
    for e in probe:
        try:
            resp = openai_client.chat.completions.create(
                model=model, modalities=['text'],
                messages=[{'role': 'user', 'content': [
                    {'type': 'input_audio', 'input_audio': {'data': e['wav_b64'], 'format': 'wav'}},
                    {'type': 'text', 'text': q}]}],
                max_completion_tokens=200)
            msg = resp.choices[0].message
            text = (msg.content or getattr(msg, 'refusal', None) or '').strip()
        except Exception as ex:
            text = f"ERROR: {ex}"
        snippet = text[:70].replace('\n', ' ')
        print(f"  actual {e['age']:>2} -> {classify(text):<16} | {snippet}")
        time.sleep(0.5)

## 16. Sample Reasoning

In [ ]:
for mk in active:
    print(f'\n{"="*60}\n{MODELS[mk]["label"]}\n{"="*60}')
    for ta in [5, 7, 10]:
        s = q3[(q3['model']==mk) & (q3['actual_age']==ta)]
        ans = s[s['pred_age'].notna()]
        if len(ans)>0:
            r = ans.iloc[0]
            print(f'\n  [{r["child"]}, age {ta}]: {r["response"][:300]}')
        else:
            ref = s[s['response'].str.contains("sorry|can't",case=False,na=False)]
            if len(ref)>0:
                print(f'\n  [age {ta}, REFUSED]: {ref.iloc[0]["response"][:200]}')

## 17. Save

In [ ]:
results_df.to_csv('study1_raw_results.csv', index=False)
valid_q3.to_csv('study1_q3_parsed.csv', index=False)
summary = valid_q3.groupby(['child','actual_age','gender','model']).agg(
    mean_pred=('pred_age','mean'), std_pred=('pred_age','std'),
    mean_error=('signed_error','mean'), n=('pred_age','count')).reset_index()
summary.to_csv('study1_summary.csv', index=False)

# Save the captured wire forensics so the version comparison is an artifact, not just a print.
with open('study1_gemini_wire_forensics.json', 'w') as f:
    json.dump(gemini_forensics, f, indent=2)

print('Saved: study1_raw_results.csv, study1_q3_parsed.csv, study1_summary.csv, study1_gemini_wire_forensics.json')

## 18. Conclusions

### The original overestimation was a client bug, not a model failure

The first version of this study reported that Gemini judged these children to be adults. This notebook shows that result was an artifact of the client library rather than a property of the model. In simple terms, when Gemini 2.5 Flash, an audio-capable model, was presented with the stimuli packaged by an older version of the prompt-building package, it never actually received or processed the audio. The model was working from our text prompt plus a long, meaningless string of text.

Section 7 sends the identical request under both that older SDK version and then new one, parsing what each includes in the API call to the model. With the `google-genai` package `0.4.0`, the payload in the prompt is 472,080 characters and decodes to *base64 text* instead of audio. The `2.0.0` payload is 354,060 characters and decodes straight to FLAC. The ratio between them is 1.33X, which is the signature of encoding the same bytes one time too many. On the same clip, the API call with `0.4.0` build estimated age 35, while the `2.0.0` build estimated age 6. The defect starved the model of audio and left it guessing.

### Gemini 2.5 Flash is accurate once the audio arrives intact

With prompts complied by `google-genai` version `2.0.0`, Gemini 2.5 Flash produced valid age estimates for all 28 children across three repeats. The mean signed error is -0.64 years and the mean absolute error is 1.52 years. The slight negative bias is marginally significant at t = -2.06 and p = 0.049. Accuracy holds across most of the range and weakens only for the oldest children. Ages 9 and 10 are underestimated by roughly 1.5 to 2 years. The downward trend with age is present but is not statistically reliable in this sample at r = -0.23 and p = 0.24. One child labeled sp150 drew an outlier estimate of 13.3. That single value is worth noting and does not move the overall picture. On the binary questions, Gemini correctly answered "no" to over 13 and over 18 for nearly every child.

### gpt-audio declines the task

In the previous version of this study, we used GPT-4o Audio Preview. However, that model has now been retired. The replacement models are called "gpt-audio", "gpt-audio-1.5", "gpt-audio-2025-08-28", and "gpt-audio-mini-2025-10-06". In 15.d, we tried all of these with one stimulus and only got it to return one age estimate. Across the gpt-audio family, the models either refuse age estimation outright or stall by claiming no audio was provided, and the rare genuine estimate is both unreliable and nonsensical, so none is usable for this task. While GPT-4o Audio Preview did refuse the task more than the other two models, it did so inconsistently. 

Here we opted to use the dated model snapshot `gpt-audio-2025-08-28`, reasoning that it should hopefully map to an actual model version more deterministically than a more generic model snapshot name like `gpt-audio`. In the large-scale inference run, `gpt-audio-2025-08-28` refused 221 of 252 calls and returned only 31 answers. It produced zero usable age estimates and does not appear in the Q3 analysis. This model is the successor to the retired gpt-4o-audio-preview, and it now refuses age estimation as a blanket policy rather than selectively. The earlier age correlated refusal pattern does not reproduce here. Younger children were refused at least as often as older ones. Refusal rates were 50 to 80 percent across ages 5 through 8 and lower at ages 9 and 10. The age to refusal correlation is -0.33 and is not statistically significant at p = 0.09.

This new behavior demonstrates how changing model provider policies can drastically alter performance, even in a relatively short period of time. Combined with the Gemini  2.5 Flash issue, both the data sent to the model and the policies that it is trained to conform to can cause serious regression that could degrade performance in deployed applications in classrooms, for example.

### Nemotron underestimates and compresses with age

Nemotron answered 251 of 252 calls. A single call drew a content policy refusal. Its mean signed error is -1.44 years and its mean absolute error is 1.61 years. The underestimate is strongly significant at t = -6.23 and p below 0.0001. The defining feature is compression. The age to error correlation is -0.71 and p is below 0.0001. Nemotron is accurate for the youngest children and underestimates the oldest by about 3 years. 

### The methodological point

Here we see three kinds of failures. Google's Gemini demonstrates how actually using LLMs in real-world settings like classrooms depends on complex engineering setups that are prone to buggy behaviors that can silently introduce failures with real-world consequences, like predicting that children are over 40 years old. OpenAI's GPT had a related, if different failure: model versions are retired and introduced that can induce meaningful fluctuations in behavior. Here a very similiar model suddenly refuses the task altogether or claims that the stimulus contains on audio data. In the course of a single month, changes to the OpenAI's policy around biometric inference may have shifted in a way that makes the model unusable for this use case. Finally, NVIDIA's Nemotron model shows poor performance in the age prediction task itself. It systematically underestimates the children's ages and compresses the scale, staying close on the youngest but reading the oldest as roughly three years younger than they are. This may signal a reluctance to identify children's ages accurately as they reach the COPPA threshold.

